In [1]:
!pip install snowflake-snowpark-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 158.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 213.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 183.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1
  Attempting uninstall: cryptography
    Found existing installation: cryptography 44.0.0
    Uninstalling cryptography-44.0.0:
      Successfully uninstalled cryptography-44.0.0


In [2]:
import pandas as pd
import numpy as np
import os
import sklearn
import sys
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, min, max, abs, avg, median, month, year, when, round, is_null, upper, trim, regexp_replace, split, sum, datediff, dateadd, to_timestamp, to_date, last_day, current_date, least, lag, lit, coalesce, greatest, count_distinct, count, expr, try_cast, array_slice, array_construct, array_compact, sort_array, concat_ws, concat
from snowflake.snowpark.window import Window

## Functions to read in Snowflake data

In [4]:
import snowflake.snowpark
print(snowflake.snowpark.__version__)

1.53.1


In [5]:
def create_snowpark_session():
    # Create session with connection parameters
    connection_parameters = {
        "account": "phrwdstore.us-east-1",
        "user": "vishnu.suresh1.ext@bayer.com", #add your email here
        "role": "RWDSTORE_TRINETX_EHR_ATTRCM_R",  # Primary role of the database
        "warehouse": "COMPUTE_WH",
        "database": "TRINETX_EHR_ATTRCM",
        #"schema": "EDCOY",
        "authenticator":"externalbrowser"
    }
    
    session = Session.builder.configs(connection_parameters).create()
    return session

In [6]:
session = create_snowpark_session()

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/fcb2b37b-5da0-466b-9b83-0014b67a7c78/saml2?SAMLRequest=nZJJb9swEIX%2FisCeJVGLN8J24Np1ayBpHS8pkBtFjW3WEqlyqCjury%2FlBUgPyaE3gnwz3%2BO8Gd69loX3AgalViMSBZR4oITOpdqPyHYz9%2FvEQ8tVzgutYEROgORuPEReFhWb1PagVvC7BrSea6SQtQ8jUhvFNEeJTPESkFnB1pOHexYHlHFEMNbhyLUkR%2BlYB2srFoZN0wRNEmizD2NKaUgHoVO1kk%2FkDaL6mFEZbbXQxa3k1f3pHUQU0rRFOIUjLK%2BFn6W6jOAjSnYRIfu22Sz95Y%2F1hniT2%2B%2BmWmFdglmDeZECtqv7iwF0DqqDaXK02kBQow8crR8FqHSzK%2FgRhC6r2rrGgTuFO8jDQu%2BlG9diNiLVUebqT0fAEuEXruZ7scf%2BIx18PX3ZPjacPlfTKMl%2BpqunbLbOjoJ4T7dw4zbcBWINC9VGat0Vjbs%2B7flxfxN1WEpZGgfdJHkm3sxFKhW358qb77OPoJTCaNQ7q1UhFVxciizOkl7md3JO%2FbTbzfxB1k98N9806%2FZ4T%2FT6YRtcTC7Lw85GzPg%2FRjIM3za4ruJ3l85ittSFFCdvrk3J7fvhRUF0vpG5vztLGZRcFpM8N4DoQiwK3UwNcOs23poaSDi%2BUP%2Fd%2BfFf&RelayState=ver%3A3-hint%3A3199608952094726-ETMsDgAAAZ%2BpYqcJABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEB3K57YfHWj14I360C%2Fa86QAAAC

Enter the URL the SSO URL redirected you to:  http://localhost:58915/?token=vVhbk6JMEv0rHc6jodwFDdtYLmqjgiKIyssGlwJRblIg6q_fQqdvMz29_e3MbIQPVSmVeTIrK_NU9aEdR1lvCWCWJhA8yNJj49-0R_k-gzMtD-foFm1zeMthgNcCHueTrNPF8S7ReDBBDsM0eWyQbbzxIENYAjmBhZ0USISTnRbOtkjOIJgejfdous0RlNV4kAAswsQubit3RZHBHoZlu7zyYJHmoF3CFrBh0SLaMEkrP7IPwE3jrESLgjYaYT7wsCgNwgTZTJ5xG-ljIzuEXnJlXLCAYA-Xo8ANIKfh3fFluNIqG7cykaCcNb00HUl3Dm7j4RxHCezdIvDYKPOkl9owhL3EjgHsFW5P55VZD3nXy_K0SN00agz6Nz_z-9LPF9kQgrz2szF49hMWsF2FiZdWsJ2AAvNdh3Qo1mkxno236E7HaXUdjmrhOEE7HdZmXZbD-tjd5qB_3yu9sIsSvp-JqQceTDsqweeY4O3rnl66LoCwgQ362Hul_DPoeyaQFEPZTpdsOR5wWjRejziGanG473ZJYDMk_rW9xzvWjxnzz0L4PfB_KpR6GKAkLHPwDKPWi9RWVdWuqHaaBxhyDcfwLoY-8GAYfGvcVwFPTvx00BftJE1C147C6y2bFVDsUu-Bj4I0D4td_AuVBEbgtcoWOLstl6CTb_UuvMD5shacfgbWitG5-ZZDuwV3Nsl0an1L4IMcJC54WC3lx8a3r23koG_kdgL9NI_hm_F_RfMuTCA5gSjNULmAz07ViL6u7eMIYW-hSWGAysg_jBUKx7fXCN1V3I7MQCS0ciZii0sFFlwepywHmW1AL4ztVPPXcqjPcGVyOM7EIffYx96u7GMvkUbjt-nxsqPfTWyJbSRdhWsCzcMsNSJ1tmWn6XGouPp0OOyaAOIbWqAh08zHU-mkdrqy